In [ ]:
# Importa as bibliotecas necessárias para manipular arquivos e tabelas
from pathlib import Path
import pandas as pd

# Define o caminho do arquivo CSV dentro da mesma pasta do notebook
# Path.cwd() pega o diretório atual de trabalho do notebook
csv_path = Path.cwd() / "mapa_salas_tidy.csv"

# Lê o arquivo CSV em um DataFrame do pandas
# O DataFrame é uma tabela tabular que facilita a análise dos dados
# Aqui, estamos assumindo que o CSV está no mesmo diretório do notebook
# e que o arquivo foi gerado corretamente em formato tabular

df = pd.read_csv(csv_path)

# Exibe informações básicas sobre a base de dados
print(f"Arquivo: {csv_path}")
print(f"Linhas: {len(df)}")
print(f"Colunas: {list(df.columns)}")

# Mostra as primeiras linhas do DataFrame para confirmar se a leitura foi bem-sucedida
# Isso ajuda a verificar se os nomes das colunas e os valores estão corretos
print(df.head())

In [6]:
# Carrega a base de dados
# A base já foi importada no notebook anterior, mas este bloco garante que ela esteja disponível
# e realiza o cálculo específico pedido: carga horária do curso ARQU no período da tarde

# Importa a biblioteca de manipulação de dados
import pandas as pd

# Garante que o DataFrame exista
# Se a célula anterior ainda não foi executada, esta linha cria o DataFrame a partir do CSV
if 'df' not in globals():
    df = pd.read_csv("mapa_salas_tidy.csv")

# Converte os horários em minutos desde a meia-noite para facilitar a comparação
# Exemplo: 13:30 -> 810; 18:30 -> 1110
# Isso permite verificar se cada turma está dentro do intervalo da tarde

def hora_para_minutos(hora_str):
    # Converte strings no formato HH:MM para minutos totais
    h, m = map(int, hora_str.split(':'))
    return h * 60 + m

# Cria colunas numéricas de horário para comparação mais simples
# A coluna 'hora_inicio' é o início do encontro, 'hora_fim' é o fim
# e 'numero_periodos' indica a duração em períodos de 1 hora

df['hora_inicio_min'] = df['hora_inicio'].apply(hora_para_minutos)
df['hora_fim_min'] = df['hora_fim'].apply(hora_para_minutos)

# Define os limites do período da tarde desejado
# Intervalo: 13:30 até 18:30
janela_inicio = 13 * 60 + 30
janela_fim = 18 * 60 + 30

# Filtra apenas o curso de Arquitetura e Urbanismo (ARQU)
# Em seguida, mantém apenas os registros que entram na janela da tarde
# A condição abaixo seleciona encontros cujo início e fim ficam dentro do intervalo
# ou que se sobrepõem parcialmente com a janela

arqu_tarde = df[
    (df['curso'] == 'ARQU') &
    (df['hora_inicio_min'] < janela_fim) &
    (df['hora_fim_min'] > janela_inicio)
].copy()

# Calcula a carga horária efetiva dentro da janela da tarde
# Para cada registro, a sobreposição é calculada como a parte do encontro dentro do intervalo
# A carga total será a soma dessas sobreposições em horas
arqu_tarde['sobreposicao_min'] = arqu_tarde.apply(
    lambda row: max(0, min(row['hora_fim_min'], janela_fim) - max(row['hora_inicio_min'], janela_inicio)),
    axis=1
)

# Soma a sobreposição em minutos e converte para horas
carga_horaria_minutos = arqu_tarde['sobreposicao_min'].sum()
carga_horaria_horas = carga_horaria_minutos / 60

# Exibe o resultado final
print('Carga horária do curso ARQU no período da tarde (13:30 às 18:30):')
print(f'{carga_horaria_horas:.2f} horas')
print(f'Equivalente a {carga_horaria_minutos} minutos')

# Se quiser, também pode visualizar quais registros compõem esse total
print('\nRegistros considerados no cálculo:')
print(arqu_tarde[['nome_disciplina', 'dia_semana', 'hora_inicio', 'hora_fim', 'sobreposicao_min']].head(20).to_string(index=False))

Carga horária do curso ARQU no período da tarde (13:30 às 18:30):
143.00 horas
Equivalente a 8580 minutos

Registros considerados no cálculo:
                                 nome_disciplina    dia_semana hora_inicio hora_fim  sobreposicao_min
                         REPRESENTAÇÃO GRÁFICA I   SEXTA-FEIRA       13:30    16:30               180
SISTEMAS DE INFORMAÇÕES GEOGRÁFICAS EM URBANISMO   TERÇA-FEIRA       14:30    18:30               240
                        REPRESENTAÇÃO GRÁFICA II SEGUNDA-FEIRA       13:30    16:30               180
                        REPRESENTAÇÃO GRÁFICA II SEGUNDA-FEIRA       13:30    16:30               180
                         REPRESENTAÇÃO GRÁFICA I   TERÇA-FEIRA       13:30    16:30               180
                         REPRESENTAÇÃO GRÁFICA I   TERÇA-FEIRA       13:30    16:30               180
                        REPRESENTAÇÃO GRÁFICA II  QUARTA-FEIRA       13:30    16:30               180
                        REPRESENTAÇÃO GRÁF

In [ ]:
# Calcula as horas livres de cada sala no turno da manhã (07:30 às 12:30).
import pandas as pd

# Usa o DataFrame já carregado ou lê o CSV, caso necessário.
if 'df' not in globals():
    df = pd.read_csv("mapa_salas_tidy.csv")

# Converte um horário HH:MM para minutos desde meia-noite.
def hora_para_minutos(hora_str):
    horas, minutos = map(int, hora_str.split(':'))
    return horas * 60 + minutos

# Define os limites do turno da manhã e sua duração total.
manhã_inicio = 7 * 60 + 30
manha_fim = 12 * 60 + 30
duracao_turno_manha = manha_fim - manha_inicio

# Une intervalos sobrepostos da mesma sala e dia.
# Isso evita contar duas vezes uma sala ocupada simultaneamente por turmas diferentes.
def minutos_ocupados(intervalos):
    total = 0
    inicio_atual = fim_atual = None

    # Ordena os intervalos para construir blocos contínuos de ocupação.
    for inicio, fim in sorted(intervalos):
        if inicio_atual is None:
            inicio_atual, fim_atual = inicio, fim
        elif inicio <= fim_atual:
            # Estende o bloco quando o novo intervalo se sobrepõe ao anterior.
            fim_atual = max(fim_atual, fim)
        else:
            # Fecha o bloco atual e inicia outro intervalo independente.
            total += fim_atual - inicio_atual
            inicio_atual, fim_atual = inicio, fim

    # Soma o último bloco construído.
    if inicio_atual is not None:
        total += fim_atual - inicio_atual
    return total

# Cria uma cópia para calcular os horários sem alterar o DataFrame original.
ocupacao = df.copy()
ocupacao['hora_inicio_min'] = ocupacao['hora_inicio'].apply(hora_para_minutos)
ocupacao['hora_fim_min'] = ocupacao['hora_fim'].apply(hora_para_minutos)

# Mantém somente os encontros que ocupam alguma parte do turno da manhã.
ocupacao = ocupacao[
    (ocupacao['hora_inicio_min'] < manha_fim) &
    (ocupacao['hora_fim_min'] > manha_inicio)
].copy()

# Recorta cada encontro para dentro dos limites da manhã.
ocupacao['inicio_manha'] = ocupacao[['hora_inicio_min']].clip(lower=manha_inicio)
ocupacao['fim_manha'] = ocupacao[['hora_fim_min']].clip(upper=manha_fim)

# Calcula os minutos ocupados por sala e por dia.
ocupacao_por_dia = (
    ocupacao.groupby(['sala', 'dia_semana'])
    .apply(
        lambda grupo: minutos_ocupados(
            zip(grupo['inicio_manha'], grupo['fim_manha'])
        ),
        include_groups=False
    )
    .rename('minutos_ocupados')
    .reset_index()
)

# Gera todas as combinações possíveis de sala e dia.
# Assim, também aparecem salas sem aulas em determinados dias.
salas = df[['sala']].drop_duplicates()
dias = pd.DataFrame({'dia_semana': sorted(df['dia_semana'].dropna().unique())})
base_salas_dias = salas.merge(dias, how='cross')

# Junta a ocupação calculada à tabela completa sala-dia.
# Quando não há aula, os minutos ocupados são preenchidos com zero.
horas_livres_manha = base_salas_dias.merge(
    ocupacao_por_dia,
    on=['sala', 'dia_semana'],
    how='left'
).fillna({'minutos_ocupados': 0})

# Subtrai o tempo ocupado da duração total do turno.
horas_livres_manha['minutos_livres'] = (
    duracao_turno_manha - horas_livres_manha['minutos_ocupados']
)
horas_livres_manha['horas_livres'] = horas_livres_manha['minutos_livres'] / 60

# Exibe as horas livres de cada sala em cada dia.
print('Horas livres por sala e dia no turno da manhã (07:30 às 12:30):')
print(
    horas_livres_manha[['sala', 'dia_semana', 'horas_livres']]
    .sort_values(['sala', 'dia_semana'])
    .to_string(index=False)
)

# Soma as horas livres de todos os dias para cada sala.
resumo_por_sala = (
    horas_livres_manha.groupby('sala', as_index=False)['horas_livres']
    .sum()
    .sort_values('sala')
)

print('\nSomatório de horas livres por sala em todos os dias:')
print(resumo_por_sala.to_string(index=False))
print(
    f"\nSomatório total de horas livres de todas as salas: "
    f"{resumo_por_sala['horas_livres'].sum():.2f} horas"
)


Horas livres por sala e dia no turno da manhã (07:30 às 12:30):
     sala    dia_semana  horas_livres
 Sala 103  QUARTA-FEIRA           2.0
 Sala 103  QUINTA-FEIRA           3.0
 Sala 103 SEGUNDA-FEIRA           2.0
 Sala 103   SEXTA-FEIRA           5.0
 Sala 103   TERÇA-FEIRA           3.0
Sala 301A  QUARTA-FEIRA           1.0
Sala 301A  QUINTA-FEIRA           2.0
Sala 301A SEGUNDA-FEIRA           1.0
Sala 301A   SEXTA-FEIRA           2.0
Sala 301A   TERÇA-FEIRA           3.0
Sala 301B  QUARTA-FEIRA           2.0
Sala 301B  QUINTA-FEIRA           2.0
Sala 301B SEGUNDA-FEIRA           5.0
Sala 301B   SEXTA-FEIRA           2.0
Sala 301B   TERÇA-FEIRA           2.0
 Sala 302  QUARTA-FEIRA           2.0
 Sala 302  QUINTA-FEIRA           2.0
 Sala 302 SEGUNDA-FEIRA           2.0
 Sala 302   SEXTA-FEIRA           2.0
 Sala 302   TERÇA-FEIRA           2.0
 Sala 303  QUARTA-FEIRA           2.0
 Sala 303  QUINTA-FEIRA           2.0
 Sala 303 SEGUNDA-FEIRA           2.0
 Sala 303   SEXTA-FEIRA 

In [5]:
# Calcula a ocupação dos cursos DVIS e DPRO no turno da manhã.
import pandas as pd

# Usa o DataFrame existente ou carrega os dados do CSV.
if 'df' not in globals():
    df = pd.read_csv("mapa_salas_tidy.csv")

# Converte horários no formato HH:MM para minutos desde meia-noite.
def hora_para_minutos(hora_str):
    horas, minutos = map(int, hora_str.split(':'))
    return horas * 60 + minutos

# Une intervalos sobrepostos e soma somente o tempo efetivamente ocupado.
# Assim, uma sala ocupada por duas turmas simultâneas não é contada duas vezes.
def minutos_ocupados(intervalos):
    total = 0
    inicio_atual = fim_atual = None

    # A ordenação permite comparar cada intervalo com o bloco anterior.
    for inicio, fim in sorted(intervalos):
        if inicio_atual is None:
            inicio_atual, fim_atual = inicio, fim
        elif inicio <= fim_atual:
            # Intervalos que se tocam ou se sobrepõem formam um único bloco.
            fim_atual = max(fim_atual, fim)
        else:
            # Fecha o bloco anterior antes de iniciar um novo.
            total += fim_atual - inicio_atual
            inicio_atual, fim_atual = inicio, fim

    # Soma o último bloco depois que todos os intervalos foram percorridos.
    if inicio_atual is not None:
        total += fim_atual - inicio_atual
    return total

# Define a janela analisada: das 07:30 às 12:30.
manha_inicio = 7 * 60 + 30
manha_fim = 12 * 60 + 30
cursos_design = ['DVIS', 'DPRO']

# Seleciona os dois cursos e somente os encontros que se cruzam com a manhã.
design_manha = df[
    (df['curso'].isin(cursos_design)) &
    (df['hora_inicio'].apply(hora_para_minutos) < manha_fim) &
    (df['hora_fim'].apply(hora_para_minutos) > manha_inicio)
].copy()

# Recorta encontros que começam antes ou terminam depois da janela analisada.
design_manha['inicio_manha'] = design_manha['hora_inicio'].apply(hora_para_minutos).clip(lower=manha_inicio)
design_manha['fim_manha'] = design_manha['hora_fim'].apply(hora_para_minutos).clip(upper=manha_fim)
design_manha['curso_relatorio'] = design_manha['curso']

# Remove repetições do mesmo curso na mesma sala, dia e horário.
# Essas repetições podem surgir quando uma turma atende mais de um curso.
design_manha = design_manha.drop_duplicates(
    subset=['curso_relatorio', 'sala', 'dia_semana', 'inicio_manha', 'fim_manha']
)

# Calcula os minutos ocupados por curso, sala e dia.
ocupacao_por_curso_sala_dia = (
    design_manha.groupby(['curso_relatorio', 'sala', 'dia_semana'])
    .apply(
        lambda grupo: minutos_ocupados(
            zip(grupo['inicio_manha'], grupo['fim_manha'])
        ),
        include_groups=False
    )
    .rename('minutos_ocupados')
    .reset_index()
)

# Soma a ocupação de todas as salas e dias para cada curso.
ocupacao_por_curso = (
    ocupacao_por_curso_sala_dia.groupby('curso_relatorio', as_index=False)['minutos_ocupados']
    .sum()
)
ocupacao_por_curso['horas_ocupadas'] = ocupacao_por_curso['minutos_ocupados'] / 60

# Calcula o total combinado dos dois cursos por sala e dia.
# Aqui, intervalos simultâneos de DVIS e DPRO são unidos para não duplicar o tempo.
ocupacao_combinada = (
    design_manha.groupby(['sala', 'dia_semana'])
    .apply(
        lambda grupo: minutos_ocupados(
            zip(grupo['inicio_manha'], grupo['fim_manha'])
        ),
        include_groups=False
    )
    .rename('minutos_ocupados')
    .reset_index()
)

# Exibe a ocupação individual de cada curso.
print('Horas ocupadas por curso no turno da manhã (07:30 às 12:30):')
print(
    ocupacao_por_curso[['curso_relatorio', 'horas_ocupadas']]
    .sort_values('curso_relatorio')
    .to_string(index=False)
)

# Exibe o total combinado, sem contar duas vezes os intervalos simultâneos.
print(
    f"\nTotal combinado de DVIS e DPRO: "
    f"{ocupacao_combinada['minutos_ocupados'].sum() / 60:.2f} horas"
)

# Mostra o detalhamento da ocupação combinada por sala e dia.
print('\nDetalhamento combinado por sala e dia:')
ocupacao_combinada['horas_ocupadas'] = ocupacao_combinada['minutos_ocupados'] / 60
print(ocupacao_combinada.to_string(index=False))


Horas ocupadas por curso no turno da manhã (07:30 às 12:30):
curso_relatorio  horas_ocupadas
           DPRO            52.0
           DVIS            37.0

Total combinado de DVIS e DPRO: 61.00 horas

Detalhamento combinado por sala e dia:
     sala    dia_semana  minutos_ocupados  horas_ocupadas
 Sala 103  QUINTA-FEIRA               120             2.0
 Sala 103   TERÇA-FEIRA               120             2.0
Sala 301A  QUARTA-FEIRA               240             4.0
Sala 301A SEGUNDA-FEIRA               240             4.0
Sala 301A   TERÇA-FEIRA               120             2.0
Sala 301B  QUINTA-FEIRA               180             3.0
Sala 301B   TERÇA-FEIRA               180             3.0
 Sala 313  QUINTA-FEIRA               120             2.0
 Sala 314  QUINTA-FEIRA               180             3.0
 Sala 401  QUARTA-FEIRA               180             3.0
 Sala 401 SEGUNDA-FEIRA               180             3.0
 Sala 402  QUINTA-FEIRA               300             5.0
 Sal

In [4]:
# Calcula a carga horária semanal de um docente.
import pandas as pd

# Usa o DataFrame já carregado no notebook ou lê o CSV, caso necessário.
if 'df' not in globals():
    df = pd.read_csv("mapa_salas_tidy.csv")

# Define o docente que será consultado.
docente = "Prof10"

# Filtra somente as turmas atribuídas ao docente escolhido.
professor = df[df["docente"].eq(docente)].copy()

# Uma turma pode aparecer mais de uma vez no CSV quando atende vários cursos.
# A remoção de duplicatas evita contar o mesmo encontro mais de uma vez.
professor = professor.drop_duplicates(
    subset=[
        "codigo_disciplina", "turma", "nome_disciplina", "sala",
        "dia_semana", "hora_inicio", "hora_fim", "numero_periodos"
    ]
)

# Converte o horário HH:MM em minutos desde meia-noite.
# Isso permite calcular a duração sem depender de operações com texto.
professor["inicio_min"] = professor["hora_inicio"].map(
    lambda hora: int(hora[:2]) * 60 + int(hora[3:])
)
professor["fim_min"] = professor["hora_fim"].map(
    lambda hora: int(hora[:2]) * 60 + int(hora[3:])
)

# Calcula a duração de cada encontro em horas.
professor["duracao_horas"] = (
    professor["fim_min"] - professor["inicio_min"]
) / 60

# Soma a duração de todos os encontros da semana.
carga_semanal = professor["duracao_horas"].sum()

# Exibe o total e os encontros que formam esse resultado.
print(f"Docente: {docente}")
print(f"Carga horária semanal: {carga_semanal:.2f} horas")
print("\nEncontros considerados:")
print(
    professor[
        [
            "dia_semana", "hora_inicio", "hora_fim", "duracao_horas",
            "codigo_disciplina", "turma", "nome_disciplina", "sala"
        ]
    ].sort_values(["dia_semana", "hora_inicio"]).to_string(index=False)
)


Docente: Prof10
Carga horária semanal: 12.00 horas

Encontros considerados:
   dia_semana hora_inicio hora_fim  duracao_horas codigo_disciplina turma                          nome_disciplina      sala
 QUARTA-FEIRA       07:30    09:30            2.0          ARQ03071     A                     COMPUTAÇÃO GRÁFICA I Sala 301A
 QUARTA-FEIRA       09:30    11:30            2.0          ARQ03071     B                     COMPUTAÇÃO GRÁFICA I Sala 301A
 QUARTA-FEIRA       13:30    15:30            2.0          ARQ03138     A COMPUTAÇÃO GRÁFICA II: DESIGN DE PRODUTO Sala 301A
SEGUNDA-FEIRA       07:30    09:30            2.0          ARQ03071     A                     COMPUTAÇÃO GRÁFICA I Sala 301A
SEGUNDA-FEIRA       09:30    11:30            2.0          ARQ03071     B                     COMPUTAÇÃO GRÁFICA I Sala 301A
SEGUNDA-FEIRA       13:30    15:30            2.0          ARQ03138     A COMPUTAÇÃO GRÁFICA II: DESIGN DE PRODUTO Sala 301A


In [ ]:
# Verifica colisões de horário entre turmas do mesmo professor.
import pandas as pd
from itertools import combinations

# Usa o DataFrame já carregado no notebook ou lê o CSV, caso necessário.
if 'df' not in globals():
    df = pd.read_csv("mapa_salas_tidy.csv")

# Cada linha do CSV pode aparecer mais de uma vez quando uma turma atende
# vários cursos. Mantemos apenas um registro físico de cada encontro.
encontros = df.drop_duplicates(
    subset=[
        "docente", "codigo_disciplina", "turma", "sala",
        "dia_semana", "hora_inicio", "hora_fim"
    ]
).copy()

# Ignora linhas sem docente, pois elas não podem participar de uma colisão.
encontros = encontros[
    encontros["docente"].notna() & encontros["docente"].ne("")
].copy()

# Converte um horário no formato HH:MM para minutos desde meia-noite.
# Com valores numéricos, a comparação entre intervalos fica mais simples.
def para_minutos(hora):
    horas, minutos = map(int, hora.split(":"))
    return horas * 60 + minutos

# Cria os limites numéricos de cada encontro.
encontros["inicio_min"] = encontros["hora_inicio"].map(para_minutos)
encontros["fim_min"] = encontros["hora_fim"].map(para_minutos)

colisoes = []

# Só é necessário comparar encontros do mesmo professor e do mesmo dia.
for (nome_docente, dia), grupo in encontros.groupby(["docente", "dia_semana"]):
    # combinations gera cada par uma única vez, evitando comparar A-B e B-A.
    for indice_a, indice_b in combinations(grupo.index, 2):
        encontro_a = grupo.loc[indice_a]
        encontro_b = grupo.loc[indice_b]

        # Dois intervalos se sobrepõem quando o maior início ocorre
        # antes do menor fim. O uso de '<' permite encontros consecutivos,
        # como 07:30-09:30 e 09:30-11:30, sem classificá-los como colisão.
        intervalo_sobreposto = (
            max(encontro_a["inicio_min"], encontro_b["inicio_min"])
            < min(encontro_a["fim_min"], encontro_b["fim_min"])
        )

        # A colisão só interessa quando o professor teria de estar
        # fisicamente em duas salas diferentes ao mesmo tempo.
        salas_diferentes = encontro_a["sala"] != encontro_b["sala"]

        if intervalo_sobreposto and salas_diferentes:
            colisoes.append({
                "docente": nome_docente,
                "dia_semana": dia,
                "hora_inicio_a": encontro_a["hora_inicio"],
                "hora_fim_a": encontro_a["hora_fim"],
                "sala_a": encontro_a["sala"],
                "turma_a": f"{encontro_a['codigo_disciplina']}-{encontro_a['turma']}",
                "hora_inicio_b": encontro_b["hora_inicio"],
                "hora_fim_b": encontro_b["hora_fim"],
                "sala_b": encontro_b["sala"],
                "turma_b": f"{encontro_b['codigo_disciplina']}-{encontro_b['turma']}",
            })

# Transforma a lista em DataFrame para facilitar a visualização e análises futuras.
colisoes_df = pd.DataFrame(colisoes)

if colisoes_df.empty:
    print("Nenhuma colisão de carga horária foi encontrada.")
else:
    print(f"Foram encontradas {len(colisoes_df)} colisões:")
    print(colisoes_df.to_string(index=False))


In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

try:
    from streamlit.runtime.scriptrunner import get_script_run_ctx

    EM_STREAMLIT = get_script_run_ctx(suppress_warning=True) is not None
except (ImportError, AttributeError):
    EM_STREAMLIT = False

if "df" in globals():
    dados_calendario = df.copy()
else:
    caminho_csv = Path.cwd() / "mapa_salas_tidy.csv"
    dados_calendario = pd.read_csv(caminho_csv, encoding="utf-8-sig")

colunas_calendario = {
    "semestre", "predio", "sala", "tipo_sala", "capacidade_turma",
    "codigo_disciplina", "turma", "nome_disciplina", "docente", "curso",
    "dia_semana", "hora_inicio", "hora_fim", "numero_periodos",
    "vagas_oferecidas", "vagas_totais_compartilhadas",
    "turmas_compartilhando_sala", "etapa",
}
colunas_faltantes = colunas_calendario.difference(dados_calendario.columns)
if colunas_faltantes:
    raise ValueError(
        "Colunas necessárias ausentes: " + ", ".join(sorted(colunas_faltantes))
    )

ORDEM_DIAS_CALENDARIO = [
    "SEGUNDA-FEIRA", "TERÇA-FEIRA", "QUARTA-FEIRA",
    "QUINTA-FEIRA", "SEXTA-FEIRA", "SÁBADO", "DOMINGO",
]
dados_calendario["etapa_num"] = pd.to_numeric(
    dados_calendario["etapa"], errors="coerce"
)
dados_calendario["etapa_label"] = dados_calendario["etapa_num"].map(
    lambda valor: (
        str(int(valor))
        if pd.notna(valor) and float(valor).is_integer()
        else str(valor) if pd.notna(valor) else ""
    )
)

if EM_STREAMLIT:
    st.set_page_config(page_title="Calendário semanal de turmas", layout="wide")
    st.title("Calendário semanal de turmas")
    st.caption("Filtre por curso, etapa ou dia. Passe o cursor sobre um encontro para ver os detalhes.")

    cursos_disponiveis = sorted(
        dados_calendario["curso"].dropna().astype(str).unique()
    )
    dias_disponiveis = [
        dia for dia in ORDEM_DIAS_CALENDARIO
        if dia in dados_calendario["dia_semana"].dropna().unique()
    ]
    coluna_curso, coluna_etapa, coluna_dia = st.columns(3)
    with coluna_curso:
        curso_selecionado = st.selectbox(
            "Curso", ["Todos os cursos", *cursos_disponiveis]
        )

    dados_para_etapas = dados_calendario.copy()
    if curso_selecionado != "Todos os cursos":
        dados_para_etapas = dados_para_etapas[
            dados_para_etapas["curso"].astype(str).eq(curso_selecionado)
        ]
    etapas_disponiveis = sorted(
        etapa for etapa in dados_para_etapas["etapa_label"].unique() if etapa
    )
    with coluna_etapa:
        etapas_selecionadas = st.multiselect(
            "Etapa", etapas_disponiveis, default=etapas_disponiveis
        )
    with coluna_dia:
        dias_selecionados = st.multiselect(
            "Dia da semana", dias_disponiveis, default=dias_disponiveis
        )

    dados_filtrados = dados_para_etapas.copy()
    if etapas_selecionadas:
        dados_filtrados = dados_filtrados[
            dados_filtrados["etapa_label"].isin(etapas_selecionadas)
        ]
    if dias_selecionados:
        dados_filtrados = dados_filtrados[
            dados_filtrados["dia_semana"].isin(dias_selecionados)
        ]
else:
    dados_filtrados = dados_calendario.copy()

# Linhas de cursos diferentes podem descrever o mesmo encontro físico.
# Agrupamos esse encontro e reunimos as informações para o hover.
CHAVE_ENCONTRO_CALENDARIO = [
    "semestre", "predio", "sala", "tipo_sala", "codigo_disciplina",
    "nome_disciplina", "dia_semana", "hora_inicio", "hora_fim",
    "turmas_compartilhando_sala",
]


def listar_valores_unicos(valores):
    valores_unicos = {
        str(valor).strip()
        for valor in valores
        if pd.notna(valor) and str(valor).strip()
    }
    return ", ".join(sorted(valores_unicos)) or "Não informado"


def formatar_vagas_por_turma(grupo):
    colunas_vagas = grupo[
        ["turma", "curso", "vagas_oferecidas"]
    ].drop_duplicates()
    descricoes = []
    for _, linha in colunas_vagas.iterrows():
        if pd.notna(linha["vagas_oferecidas"]):
            descricoes.append(
                f"Turma {linha['turma']} ({linha['curso']}): "
                f"{linha['vagas_oferecidas']:g}"
            )
    return "<br>".join(sorted(set(descricoes))) or "Não informado"


eventos = []
for _, grupo in dados_filtrados.groupby(
    CHAVE_ENCONTRO_CALENDARIO, dropna=False, sort=False
):
    linha_base = grupo.iloc[0]
    eventos.append(
        {
            **{coluna: linha_base[coluna] for coluna in CHAVE_ENCONTRO_CALENDARIO},
            "cursos_hover": listar_valores_unicos(grupo["curso"]),
            "etapas_hover": listar_valores_unicos(grupo["etapa_label"]),
            "turmas_hover": listar_valores_unicos(grupo["turma"]),
            "docentes_hover": listar_valores_unicos(grupo["docente"]),
            "capacidade_hover": listar_valores_unicos(grupo["capacidade_turma"]),
            "vagas_turma_hover": formatar_vagas_por_turma(grupo),
            "vagas_compartilhadas_hover": listar_valores_unicos(
                grupo["vagas_totais_compartilhadas"]
            ),
            "periodos": linha_base["numero_periodos"],
        }
    )

encontros_calendario = pd.DataFrame(eventos)


def horario_em_minutos(horario):
    horas, minutos = map(int, str(horario).split(":"))
    return horas * 60 + minutos


figura_calendario = go.Figure()
if not encontros_calendario.empty:
    encontros_calendario["inicio_minutos"] = encontros_calendario[
        "hora_inicio"
    ].map(horario_em_minutos)
    encontros_calendario["fim_minutos"] = encontros_calendario[
        "hora_fim"
    ].map(horario_em_minutos)
    encontros_calendario["duracao_minutos"] = (
        encontros_calendario["fim_minutos"]
        - encontros_calendario["inicio_minutos"]
    )
    encontros_calendario["dia_indice"] = encontros_calendario[
        "dia_semana"
    ].map({dia: indice for indice, dia in enumerate(ORDEM_DIAS_CALENDARIO)})
    encontros_calendario = encontros_calendario.dropna(
        subset=["dia_indice", "inicio_minutos", "fim_minutos"]
    ).copy()
    encontros_calendario["dia_indice"] = encontros_calendario[
        "dia_indice"
    ].astype(int)

    # Distribui encontros simultâneos em faixas lado a lado para não ocultá-los.
    encontros_calendario["faixa"] = 0
    faixas_por_dia = {}
    for indice_dia, grupo_dia in encontros_calendario.groupby("dia_indice"):
        finais_faixas = []
        ordem = grupo_dia.sort_values(
            ["inicio_minutos", "fim_minutos"]
        ).index
        for indice in ordem:
            inicio = encontros_calendario.at[indice, "inicio_minutos"]
            fim = encontros_calendario.at[indice, "fim_minutos"]
            faixa_livre = next(
                (i for i, final in enumerate(finais_faixas) if final <= inicio),
                len(finais_faixas),
            )
            if faixa_livre == len(finais_faixas):
                finais_faixas.append(fim)
            else:
                finais_faixas[faixa_livre] = fim
            encontros_calendario.at[indice, "faixa"] = faixa_livre
        faixas_por_dia[indice_dia] = len(finais_faixas)

    encontros_calendario["posicao_x"] = encontros_calendario.apply(
        lambda linha: (
            linha["dia_indice"] - 0.45
            + (linha["faixa"] + 0.5) * 0.9
            / faixas_por_dia[linha["dia_indice"]]
        ),
        axis=1,
    )
    encontros_calendario["largura"] = encontros_calendario["dia_indice"].map(
        lambda dia: 0.86 / faixas_por_dia[dia]
    )
    encontros_calendario["rotulo"] = (
        encontros_calendario["codigo_disciplina"]
        + " - " + encontros_calendario["turmas_hover"]
    )

    cores = px.colors.qualitative.Set3
    codigos = sorted(encontros_calendario["codigo_disciplina"].unique())
    cores_por_codigo = {
        codigo: cores[indice % len(cores)]
        for indice, codigo in enumerate(codigos)
    }
    dados_hover = encontros_calendario[
        [
            "codigo_disciplina", "nome_disciplina", "cursos_hover",
            "etapas_hover", "turmas_hover", "docentes_hover", "predio",
            "sala", "tipo_sala", "capacidade_hover", "vagas_turma_hover",
            "vagas_compartilhadas_hover", "periodos", "hora_inicio", "hora_fim",
        ]
    ].to_numpy()
    figura_calendario.add_trace(
        go.Bar(
            x=encontros_calendario["posicao_x"],
            y=encontros_calendario["duracao_minutos"],
            base=encontros_calendario["inicio_minutos"],
            width=encontros_calendario["largura"],
            text=encontros_calendario["rotulo"],
            textposition="inside",
            insidetextanchor="middle",
            textfont={"size": 9},
            marker_color=[
                cores_por_codigo[codigo]
                for codigo in encontros_calendario["codigo_disciplina"]
            ],
            customdata=dados_hover,
            hovertemplate=(
                "<b>%{customdata[0]} - %{customdata[4]}</b><br>"
                "Disciplina: %{customdata[1]}<br>"
                "Curso(s): %{customdata[2]}<br>"
                "Etapa(s): %{customdata[3]}<br>"
                "Docente(s): %{customdata[5]}<br>"
                "Prédio / sala: %{customdata[6]} / %{customdata[7]}<br>"
                "Tipo de sala: %{customdata[8]}<br>"
                "Capacidade da turma: %{customdata[9]}<br>"
                "Vagas oferecidas: %{customdata[10]}<br>"
                "Vagas compartilhadas: %{customdata[11]}<br>"
                "Períodos: %{customdata[12]}<br>"
                "Horário: %{customdata[13]} - %{customdata[14]}"
                "<extra></extra>"
            ),
            name="Encontros",
        )
    )

horarios = list(range(450, 1351, 60))
figura_calendario.update_layout(
    title="Encontros por dia e horário",
    height=760,
    margin={"l": 65, "r": 25, "t": 65, "b": 45},
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    hoverlabel={"align": "left"},
    xaxis={
        "title": "Dia da semana",
        "tickmode": "array",
        "tickvals": list(range(len(ORDEM_DIAS_CALENDARIO))),
        "ticktext": ORDEM_DIAS_CALENDARIO,
        "range": [-0.55, len(ORDEM_DIAS_CALENDARIO) - 0.45],
        "side": "top",
        "showgrid": True,
        "gridcolor": "#e5e7eb",
        "zeroline": False,
    },
    yaxis={
        "title": "Horário",
        "tickmode": "array",
        "tickvals": horarios,
        "ticktext": [f"{minuto // 60:02d}:{minuto % 60:02d}" for minuto in horarios],
        "range": [1350, 450],
        "showgrid": True,
        "gridcolor": "#e5e7eb",
        "zeroline": False,
    },
    barmode="overlay",
)

if EM_STREAMLIT:
    st.plotly_chart(figura_calendario, use_container_width=True)
    st.caption(f"{len(encontros_calendario)} encontros no recorte selecionado.")
else:
    from IPython.display import display

    display(figura_calendario)
